# Final Pipeline: D-FINE-L + Test-Time Post-processing

Starts from the D-FINE-L model trained in `06_dfine.ipynb` (`dfine_l_896x1600_ds`, final-epoch EMA weights:
0.2987 val mAP@[.5:.95], 0.343 public LB) and adds post-processing that needs no retraining:

1. **Flip test-time augmentation**: each view is also predicted mirrored; the two predictions are fused with
   weighted boxes fusion (WBF).
2. **Tiled inference**: besides the full frame, the frame is cut into a 2x2 grid of overlapping tiles, each
   letterboxed to the model input, so objects appear ~1.7-2x larger. Small objects are the weakest point of the model
   (AP_small 0.15 vs AP_large 0.49), and training with `RandomZoomCrop` (crops of 50-100% of the frame side) already
   showed the model objects at this magnification.
3. **Temporal rescoring**: validation and test frames are consecutive video frames. Real objects persist between
   neighbouring frames while many false positives flicker, so every detection's score is mixed with the score of its
   best match in the neighbouring frames.

Each step is evaluated on the validation split (sequence-held-out, same protocol as the other notebooks) before the
test submission is built.

Sections:
1. Setup
2. Configuration
3. Model
4. Validation passes
5. Spatial post-processing on validation (flip, tiles)
6. Temporal rescoring on validation
7. Visual check
8. Test predictions and submission

## 1. Setup

Set `RUNTIME` to `"colab"` (checkpoints on Google Drive) or `"local"` (checkpoints in `runs/`). The trained run from
`06_dfine.ipynb` must be in `<storage>/dfine/<RUN_NAME>`.

In [ ]:
RUNTIME = "local"  # "colab" or "local"
REPO_URL = "https://github.com/rudkoLA/ai-hw-2-rudko-omelianchuk.git"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if RUNTIME == "colab":
    repo_dir = Path("/content/ai-hw-2-rudko-omelianchuk")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    uploaded_key = Path("/content/kaggle.json")
    if uploaded_key.exists():
        kaggle_key = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_key.parent.mkdir(parents=True, exist_ok=True)
        kaggle_key.write_bytes(uploaded_key.read_bytes())
        kaggle_key.chmod(0o600)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers", "ultralytics", "pycocotools", "kaggle"],
        check=True,
    )

print(f"Working directory: {Path.cwd()}")

In [ ]:
import hashlib
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pycocotools.coco import COCO

from src.dfine import load_dfine, load_dfine_last
from src.pipeline import (
    cached, combine, evaluate_dets, predict_passes, select, temporal_rescore, test_frames, val_frames,
    write_submission,
)
from src.setup_utils import ensure_dataset, resolve_storage, setup_environment
from src.submission import submit_to_kaggle, validate_submission

env = setup_environment(seed=42, mount_gdrive=False)
device = env["device"]
dataset_dir, coco_train, coco_val, class_names = ensure_dataset()
del coco_train, coco_val
storage = resolve_storage(RUNTIME, "dfine")

## 2. Configuration

- `WEIGHTS`: `"last"` (final epoch, 0.343 public LB) or `"best"` (best validation epoch, 0.33 public LB).
- `PIPE`: tile grid and overlap (fraction of a tile side), detections kept per view, the margin for dropping tile
  detections cut by an inner tile edge, IoU thresholds of the flip fusion and of the full-frame/tile merge, and the
  final cap of 500 detections per frame (the scorer ignores the rest).
- `EVAL_STRIDE`: metrics use every 3rd validation frame, as in `06_dfine.ipynb`, so the "full frame" row reproduces
  that notebook's number. Predictions are made for **every** validation frame so that temporal rescoring has the
  same frame-to-frame neighbours as on the (consecutive) test frames.
- Every prediction pass is cached in `<run>/pipeline/<WEIGHTS>/`, so re-running the notebook only redoes cheap steps.

In [ ]:
RUN_NAME = "dfine_l_896x1600_ds"  # trained in 06_dfine.ipynb
WEIGHTS = "last"                  # "last" or "best"

PIPE = {
    "grid": (2, 2),          # tile rows x cols
    "overlap": 0.2,          # tile overlap, fraction of a tile side
    "full_max_det": 500,     # detections kept per full-frame view
    "tile_max_det": 300,     # detections kept per tile view
    "border_margin": 2.0,    # drop tile boxes within this many px of an inner tile edge (None keeps them)
    "flip_iou": 0.55,        # WBF IoU threshold, flip pairs
    "merge": "wbf",          # full frame + tiles merge: "wbf" (max score, one box per view) or "nms"
    "merge_iou": 0.55,
    "max_det": 500,          # detections per frame in the final output
}
EVAL_STRIDE = 3
BATCH = 4    # views per forward pass
WORKERS = 4

## 3. Model

The input size and AMP setting come from the training config of the run.

In [ ]:
run_dir = storage / RUN_NAME
train_cfg = json.loads((run_dir / "cfg.json").read_text())
IMGSZ, AMP = train_cfg["imgsz"], train_cfg["amp"]
model = load_dfine(run_dir / "best", device) if WEIGHTS == "best" else load_dfine_last(run_dir, device)
pipe_dir = run_dir / "pipeline" / WEIGHTS
print(f"input {IMGSZ}, cache {pipe_dir}")


def passes_for(split, frames, keys):
    return predict_passes(
        model, frames, dataset_dir, IMGSZ, device, pipe_dir, split, keys=keys, grid=PIPE["grid"],
        overlap=PIPE["overlap"], full_max_det=PIPE["full_max_det"], tile_max_det=PIPE["tile_max_det"],
        border_margin=PIPE["border_margin"], batch=BATCH, workers=WORKERS, amp=AMP,
    )


def combo(split, passes, **kw):
    """combine() with the PIPE settings, cached by its settings."""
    kw = {"flip_iou": PIPE["flip_iou"], "merge": PIPE["merge"], "merge_iou": PIPE["merge_iou"],
          "max_det": PIPE["max_det"], **kw}
    key = hashlib.md5(json.dumps({**PIPE, **kw}, sort_keys=True, default=str).encode()).hexdigest()[:10]
    return cached(pipe_dir / f"{split}_combo_{key}.npz", lambda: combine(passes, **kw))

## 4. Validation passes

Four passes over all 2,846 validation frames: full frame, full frame flipped, 2x2 tiles, tiles flipped
(1 + 1 + 4 + 4 = 10 forward passes per frame). The tile passes take about 4x as long as a full-frame pass.

In [ ]:
vframes = val_frames(dataset_dir)
coco_val = COCO(str(dataset_dir / "val" / "annotations_coco.json"))
eval_ids = [f["id"] for f in vframes if (f["frame_no"] - 1) % EVAL_STRIDE == 0]
# test-like subset: 66% of test frames are <= 1360 px wide, while 34% of val frames are 4K
eval_ids_lowres = [f["id"] for f in vframes if (f["frame_no"] - 1) % EVAL_STRIDE == 0 and f["width"] <= 1360]
print(f"{len(vframes):,} val frames predicted, {len(eval_ids):,} evaluated ({len(eval_ids_lowres):,} of them <= 1360 px wide)")

val_passes = passes_for("val", vframes, ["full", "full_flip", "tiles", "tiles_flip"])

## 5. Spatial post-processing on validation

How the pieces are merged:
- **Flip**: each view is fused with its mirrored prediction by WBF. A cluster holds at most one box from each of the
  two predictions (two overlapping boxes of the same prediction are different objects, e.g. people in a crowd), and
  its score is the average over both, so a box found only once keeps half its score.
- **Tiles**: full-frame and tile boxes are fused by WBF across views only (at most one box per view per cluster),
  keeping the highest score. Averaging would penalise small objects that only a tile finds. Tile boxes touching an
  inner tile edge are dropped beforehand, since they are usually objects cut in half by the tile; the full frame or the
  neighbouring tile sees them whole. Plain per-class NMS is shown for comparison; unlike WBF it also suppresses
  overlapping objects within one view.

Two metric sets: all evaluated validation frames, and only frames <= 1360 px wide, which are closer to the test
set's resolution mix (validation has 4K frames, where tiling naturally helps the most).

In [ ]:
METRICS = ["mAP_50_95", "mAP_50", "mAP_75", "mAP_small", "mAP_medium", "mAP_large", "mAR_small"]


def score_row(name, d):
    m = evaluate_dets(d, vframes, coco_val, eval_ids)
    low = evaluate_dets(d, vframes, coco_val, eval_ids_lowres)
    return {"pipeline": name, **{k: m[k] for k in METRICS},
            "mAP (<=1360px)": low["mAP_50_95"], "AP_small (<=1360px)": low["mAP_small"]}


SPATIAL = {
    "full frame (06 baseline)": {},
    "+ flip": {"flip": True},
    "+ tiles (NMS)": {"tiles": True, "merge": "nms"},
    "+ tiles (WBF)": {"tiles": True},
    "+ tiles + flip": {"tiles": True, "flip": True},
}
val_sets, rows = {}, []
for name, kw in SPATIAL.items():
    val_sets[name] = combo("val", val_passes, **kw)
    rows.append(score_row(name, val_sets[name]))
spatial_table = pd.DataFrame(rows).set_index("pipeline")
spatial_table.round(4)

## 6. Temporal rescoring on validation

New score = `(1 - weight) * score + weight * mean support`, where the support from each neighbouring frame within
`window` frames is the best score among same-class boxes overlapping the detection with IoU >= `iou`.
The settings are evaluated on the best spatial setting from section 5 (by mAP on the test-like <= 1360 px frames);
the default setting is also applied to the plain full-frame predictions, to show the effect of temporal rescoring
alone. `window = 0` rows are the inputs without rescoring. Each row takes about a minute of COCO evaluation.

In [ ]:
TEMPORAL_DEFAULT = {"window": 2, "iou_thr": 0.5, "weight": 0.5}
TEMPORAL_GRID = [  # weight at window 2, then the window at weight 0.5
    {"window": 2, "iou_thr": 0.5, "weight": 0.3}, TEMPORAL_DEFAULT, {"window": 2, "iou_thr": 0.5, "weight": 0.7},
    {"window": 1, "iou_thr": 0.5, "weight": 0.5}, {"window": 4, "iou_thr": 0.5, "weight": 0.5},
]
BASELINE = "full frame (06 baseline)"
spatial_best = spatial_table["mAP (<=1360px)"].idxmax()
print(f"spatial setting for the sweep: {spatial_best}")


def temporal_row(base, t):
    d = temporal_rescore(val_sets[base], vframes, **t) if t else val_sets[base]
    t = t or {"window": 0, "weight": 0.0}
    return {"base": base, "window": t["window"], "weight": t["weight"], **score_row(base, d)}


rows = [temporal_row(BASELINE, None), temporal_row(BASELINE, TEMPORAL_DEFAULT)]
if spatial_best != BASELINE:
    rows.append(temporal_row(spatial_best, None))
rows += [temporal_row(spatial_best, t) for t in TEMPORAL_GRID if spatial_best != BASELINE or t != TEMPORAL_DEFAULT]
temporal_table = pd.DataFrame(rows).drop(columns="pipeline").set_index(["base", "window", "weight"])
display(temporal_table.round(4))

best = temporal_table.loc[spatial_best, "mAP (<=1360px)"].idxmax()
TEMPORAL = {"window": int(best[0]), "iou_thr": 0.5, "weight": float(best[1])}
if TEMPORAL["window"] == 0:
    print(f"temporal rescoring does not improve {spatial_best} on validation")
else:
    print(f"best temporal setting on {spatial_best}: {TEMPORAL}")

## 7. Visual check

Ground truth (green) and detections with score >= 0.3 of the full-frame baseline and of the best spatial setting,
on a crop of a validation frame from a 1344x756 sequence (the resolution closest to most test frames).

In [ ]:
SHOW_SCORE = 0.3
frame_i = next(i for i, f in enumerate(vframes) if f["width"] == 1344 and f["id"] in set(eval_ids))
f = vframes[frame_i]
img = cv2.cvtColor(cv2.imread(str(dataset_dir / f["file_name"])), cv2.COLOR_BGR2RGB)
gt = [a["bbox"] for a in coco_val.loadAnns(coco_val.getAnnIds(imgIds=[f["id"]], iscrowd=False))]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, name in zip(axes, ["full frame (06 baseline)", spatial_best]):
    d = val_sets[name]
    d = select(d, (d["frame"] == frame_i) & (d["score"] >= SHOW_SCORE))
    ax.imshow(img)
    for x, y, w, h in gt:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", lw=1))
    for x1, y1, x2, y2 in zip(d["x1"], d["y1"], d["x2"], d["y2"]):
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", lw=1))
    ax.set_title(f"{name}: {len(d['score'])} detections >= {SHOW_SCORE}, {len(gt)} GT")
    ax.set_xlim(0, f["width"] * 0.6)
    ax.set_ylim(f["height"] * 0.6, 0)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Test predictions and submission

Set `FINAL` from the tables above. Only the passes `FINAL` needs are predicted (and cached). The CSV is written to
`<run>/pipeline/<WEIGHTS>/submission_<setting>.csv`.

In [ ]:
FINAL = {"flip": True, "tiles": True, "temporal": True}  # choose from sections 5 and 6
FINAL_TEMPORAL = TEMPORAL                                 # best on validation, or e.g. TEMPORAL_DEFAULT
FINAL["temporal"] = FINAL["temporal"] and FINAL_TEMPORAL["window"] > 0

tframes = test_frames(dataset_dir)
keys = ["full"] + (["full_flip"] if FINAL["flip"] else []) + (["tiles"] if FINAL["tiles"] else [])
keys += ["tiles_flip"] if FINAL["flip"] and FINAL["tiles"] else []
test_passes = passes_for("test", tframes, keys)
test_dets = combo("test", test_passes, flip=FINAL["flip"], tiles=FINAL["tiles"])
if FINAL["temporal"]:
    test_dets = temporal_rescore(test_dets, tframes, **FINAL_TEMPORAL)

tag = "_".join(k for k, v in FINAL.items() if v) or "full"
if FINAL["temporal"]:
    tag += f"_w{FINAL_TEMPORAL['window']}_a{FINAL_TEMPORAL['weight']}"
submission_path = pipe_dir / f"submission_{tag}.csv"
write_submission(test_dets, tframes, submission_path, max_det=PIPE["max_det"])
report = validate_submission(submission_path, tframes)
report

Set `SUBMIT = True` to upload the file. Kaggle credentials are read from `~/.kaggle/kaggle.json` or from the
`KAGGLE_API_TOKEN` environment variable.

In [ ]:
SUBMIT = False
SUBMISSION_MESSAGE = f"D-FINE-L {RUN_NAME} ({WEIGHTS}) + {tag}"

if SUBMIT:
    assert report["is_valid"], report["errors"]
    submit_to_kaggle(submission_path, SUBMISSION_MESSAGE)